# 30 Self-Only Split-Routed High-Res/SAHI Policy MoE

29でclient expert加算が悪化したので、自己生成のみの強いrouted MoEへ戻り、scene/day-nightをrouterとして解像度とSAHIを切り替える。外部COCO expertは使わない。

In [1]:
from __future__ import annotations

import csv
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

PROJECT_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa"
RUNNER = PROJECT_ROOT / "aggressive_dqamox" / "scripts" / "build_eval_30_split_policy_highres_sahi_moe.py"
WORKSPACE = PROJECT_ROOT / "aggressive_dqamox" / "output" / "30_split_policy_highres_sahi_moe"
LOG_DIR = PROJECT_ROOT / "aggressive_dqamox" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"30_split_policy_highres_sahi_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.log"

cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace", str(WORKSPACE),
    "--target-map50", "0.55",
    "--previous-best-map50", "0.52939",
    "--gate-images", "360",
    "--min-gate-gain", "0.008",
    "--conf-thres", "0.001",
    "--tile-batch-size", "8",
]

print(" ".join(cmd))
print("log:", LOG_PATH)
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT)
print("returncode:", proc.returncode)
print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
if proc.returncode not in (0, 2):
    raise SystemExit(proc.returncode)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/scripts/build_eval_30_split_policy_highres_sahi_moe.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/30_split_policy_highres_sahi_moe --target-map50 0.55 --previous-best-map50 0.52939 --gate-images 360 --min-gate-gain 0.008 --conf-thres 0.001 --tile-batch-size 8
log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/logs/30_split_policy_highres_sahi_20260512_083833.log


returncode: 2
2%|████████▏ | 294/360 [00:24<00:15,  4.22it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  82%|████████▏ | 295/360 [00:24<00:15,  4.21it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  82%|████████▏ | 296/360 [00:25<00:15,  4.23it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  82%|████████▎ | 297/360 [00:25<00:14,  4.23it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  83%|████████▎ | 298/360 [00:25<00:14,  4.25it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  83%|████████▎ | 299/360 [00:25<00:14,  4.26it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  83%|████████▎ | 300/360 [00:25<00:14,  4.26it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  84%|████████▎ | 301/360 [00:26<00:13,  4.23it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024:  84%|████████▍ | 302/360 [00:26<00:13,  4.24it/s]
gate_hardnight_sahi640_rest1024_hardnight_sahi640_rest1024

In [2]:
from __future__ import annotations

import csv
from pathlib import Path

metrics_path = WORKSPACE / "stats" / "30_split_policy_highres_sahi_metrics.csv"
manifest_path = WORKSPACE / "stats" / "30_split_policy_highres_sahi_manifest.json"
summary_path = PROJECT_ROOT / "aggressive_dqamox" / "reports" / "30_split_policy_highres_sahi_summary.csv"

rows = list(csv.DictReader(metrics_path.open(encoding="utf-8"))) if metrics_path.exists() else []
total_rows = [r for r in rows if r.get("split") == "scene_daynight_total"]
gate_rows = [r for r in total_rows if str(r.get("phase", "")).startswith("gate_")]
full_rows = [r for r in total_rows if str(r.get("phase", "")).startswith("full_")]
best_gate = max(gate_rows, key=lambda r: (float(r.get("map50") or 0), float(r.get("map50_95") or 0))) if gate_rows else {}
best_full = max(full_rows, key=lambda r: (float(r.get("map50") or 0), float(r.get("map50_95") or 0))) if full_rows else {}

print("metrics:", metrics_path)
print("manifest:", manifest_path)
print("summary:", summary_path)
print("best gate:", best_gate.get("candidate"), best_gate.get("map50"), best_gate.get("map50_95"))
if best_full:
    print("best full:", best_full.get("candidate"), best_full.get("map50"), best_full.get("map50_95"))
else:
    print("full evaluation was skipped by the gate rule")


metrics: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/30_split_policy_highres_sahi_moe/stats/30_split_policy_highres_sahi_metrics.csv
manifest: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/30_split_policy_highres_sahi_moe/stats/30_split_policy_highres_sahi_manifest.json
summary: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/30_split_policy_highres_sahi_summary.csv
best gate: all_full1152_iou055 0.598599 0.331835
full evaluation was skipped by the gate rule
